# Quantitative Findings
How do user messages change from turn 1 to turn 5? Uses all 432 conversations (2,160 user messages) in data/wildchat_cleaned_exactly5.json. Sentiment-based checks will use the 316 scored conversations.

In [1]:
import pandas as pd
from pathlib import Path

Path.cwd()

PosixPath('/Users/jeannettejasmine/Documents/GitHub/wild-data-team/notebooks')

## 1. Message Length By Turn
**Interpretation:** The typical (median) message drops from 21 words at turn 1 to about 15 words at turns 2–5. The turn-1 mean (278 words) is much higher than the median because a few first messages are very long pasted texts (stories, instructions, code).

**Why It Matters:** turn 1 is a different kind of message from later turns: it is where users paste their setup. Comparisons between turn 1 and turn 5 should account for this difference in length, which is why the word-usage check below is repeated on short messages only.

In [2]:
# Load cleaned data
df = pd.read_json("../data/wildchat_cleaned_exactly5.json")

# Make one row per user message instead of one row per conversation
rows = []
for conversation_index, row in df.iterrows():
    for turn, text in enumerate(row["user_messages"], start=1):
        rows.append({"conversation_index": conversation_index, "turn": turn, "text": text})

messages = pd.DataFrame(rows)

# Count words in each message
messages["n_words"] = messages["text"].str.split().str.len()

# Median and mean message length at each turn
messages.groupby("turn")["n_words"].agg(["median", "mean"])

,median,mean
turn,,
1,21.0,278.041667
2,15.0,83.594907
3,15.0,78.254630
4,15.0,85.738426
5,14.0,66.912037


## 2. Longest First Messages
**Interpretation:** To see why the turn-1 mean is so high, I printed the start of the five longest turn-1 messages. Most are pasted creative-writing setups (stories, fictional forum threads). The longest (20,246 words) appears to be an automated AI-agent prompt rather than a person typing.

**Why It Matters:** some WildChat "users" may be programs, not people. Their messages say nothing about how humans feel toward AI, so this is a limitation of using WildChat to study user tone.

In [3]:
# Show the 5 longest turn-1 messages (first 200 characters of each)
longest = messages[messages["turn"] == 1].sort_values("n_words", ascending=False).head(5)

for _, row in longest.iterrows():
    print(row["n_words"], "words:", row["text"][:200])
    print("---")

20246 words: CONSTRAINTS:

1. ~100k word limit for short term memory. Your short term memory is short, so immediately save important information to files.
2. If you are unsure how you previously did something or w
---
12569 words: Title: Silverburn AU: Tesco Grand Opening & Upcoming Construction

Thread Started by ShopaholicSuzie on June 15, 2002

ShopaholicSuzie:
Hey everyone on PollokShoppers! 😍 It’s official - the new Tesco 
---
6751 words: Gloopy is a teal blob of slime, who’s malleable nature allows him to shape-shift

Glitter is a sparkling fuschia blob, and she is his best friend

They are both aliens. Write a story where she decides
---
6220 words: Note: This is in an alternate universe, This shop, despite being awesome, never existed, It does in this alternate universe, Also sorry for constantly showing you this, I’m just trying to make sure it
---
5563 words: The Chrono-Questers: Time Travelers in Tiny Boots
Chapter 1: The Magic in the Attic
- Scene 1: Alex and Emma, rummagin

## 3. Very Short Messages
**Interpretation:** Messages of 3 words or fewer (e.g., "continue," "next chapter") stay at about 7–9% at every turn. Later messages do not turn into mostly one-word follow-ups; turn 1 is simply unusually long.

**Why It Matters:** This rules out one simple explanation for the tone change: later turns are not more negative just because messages become tiny.

In [4]:
# Mark messages that are 3 words or fewer
messages["is_short"] = messages["n_words"] <= 3

# What share of messages are short at each turn?
messages.groupby("turn")["is_short"].mean()

turn
1    0.083333
2    0.074074
3    0.081019
4    0.071759
5    0.087963
Name: is_short, dtype: float64

## 4. Word Usage
**Interpretation:** Turn-1 messages are dominated by task-setup words ("write," "make," "create," "help"). By turn 5 these drop and follow-up words like "continue" and "rewrite" grow.

**Why It Matters:** Conversations shift from starting a task to revising it. The standard filler-word list removes words like "no," "not," and "again," so these correction words are checked directly below.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

In [6]:
# Set up the word counter
vectorizer = CountVectorizer(stop_words="english", binary=True, min_df=5)

# Count which words appear in each message
X = vectorizer.fit_transform(messages["text"])

# Turn it into a table: one row per message, one column per word
words = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
words["turn"] = messages["turn"]

# Share of messages containing each word, at each turn
share = words.groupby("turn").mean().T

In [7]:
share["change"] = share[5] - share[1]

print("Most common words at turn 1:")
print(share[1].sort_values(ascending=False).head(15))

print("\nMost common words at turn 5:")
print(share[5].sort_values(ascending=False).head(15))

print("\nWords that grew the most from turn 1 to turn 5:")
print(share["change"].sort_values(ascending=False).head(10))

Most common words at turn 1:
write        0.203704
make         0.136574
like         0.108796
time         0.108796
using        0.099537
new          0.094907
create       0.092593
use          0.090278
10           0.081019
help         0.078704
following    0.078704
know         0.076389
return       0.076389
world        0.074074
text         0.071759
Name: 1, dtype: float64

Most common words at turn 5:
write        0.120370
make         0.071759
use          0.060185
following    0.050926
new          0.048611
time         0.048611
data         0.043981
text         0.043981
just         0.041667
continue     0.041667
need         0.039352
different    0.039352
return       0.037037
does         0.037037
10           0.037037
Name: 5, dtype: float64

Words that grew the most from turn 1 to turn 5:
continue      0.011574
government    0.011574
rewrite       0.011574
meaning       0.011574
america       0.009259
society       0.009259
teams         0.009259
storage       0.009259


## 5. Correction and Feedback Words, Controlling for Length
**Interpretation:** Across all messages, correction words appeared to decrease, but long pasted turn-1 texts contain almost any word. Among messages of 50 words or fewer, feedback words increase from turn 1 to turn 5: "not" (4.1% → 7.3%), "again" (0% → 1.4%), "wrong" (0% → 0.8%), "continue" (0% → 2.8%), and "rewrite" (0.3% → 2.0%).

**Why It Matters:** By turn 5, users are reacting to ChatGPT's answers by correcting and redirecting it. This may explain part of the team's slightly more negative later-turn tone. Counts are small (e.g., "wrong" ≈ 3 messages), so this is suggestive, not conclusive.

In [8]:
check_words = ["no", "not", "wrong", "again", "fix", "continue", "rewrite", "thanks", "thank"]

for word in check_words:
    # \b means "whole word only", so "no" won't match "know"
    contains = messages["text"].str.lower().str.contains(rf"\b{word}\b")
    by_turn = contains.groupby(messages["turn"]).mean()
    print(f"{word:10} turn 1: {by_turn[1]:.3f}   turn 5: {by_turn[5]:.3f}")

no         turn 1: 0.100   turn 5: 0.042
not        turn 1: 0.164   turn 5: 0.111
wrong      turn 1: 0.019   turn 5: 0.012
again      turn 1: 0.051   turn 5: 0.019
fix        turn 1: 0.021   turn 5: 0.009
continue   turn 1: 0.030   turn 5: 0.042
rewrite    turn 1: 0.012   turn 5: 0.023
thanks     turn 1: 0.014   turn 5: 0.005
thank      turn 1: 0.019   turn 5: 0.007


In [9]:
# Keep only messages with 50 words or fewer
short_msgs = messages[messages["n_words"] <= 50]

# How many short messages are at each turn?
print(short_msgs.groupby("turn").size())
print()

for word in check_words:
    contains = short_msgs["text"].str.lower().str.contains(rf"\b{word}\b")
    by_turn = contains.groupby(short_msgs["turn"]).mean()
    print(f"{word:10} turn 1: {by_turn[1]:.3f}   turn 5: {by_turn[5]:.3f}")

turn
1    291
2    329
3    342
4    335
5    356
dtype: int64

no         turn 1: 0.014   turn 5: 0.020
not        turn 1: 0.041   turn 5: 0.073
wrong      turn 1: 0.000   turn 5: 0.008
again      turn 1: 0.000   turn 5: 0.014
fix        turn 1: 0.007   turn 5: 0.003
continue   turn 1: 0.000   turn 5: 0.028
rewrite    turn 1: 0.003   turn 5: 0.020
thanks     turn 1: 0.000   turn 5: 0.003
thank      turn 1: 0.000   turn 5: 0.006
